# LangChain Agents Intro

- LangChain is one of the most popular open source libraries for AI Engineers.

- It's goal is to abstract away the complexity in building AI software, provide easy-to-use building blocks, and make it easier when switching between AI service providers.

- In this example, we will introduce LangChain's Agents, adding the ability to use tools such as search and calculators to complete tasks that normal LLMs cannot fufil.


In [1]:
import json
import os
import warnings
from pathlib import Path
from typing import Any, Generator, Iterable, Type, TypeVar

# Standard imports
import numpy as np
import pandas as pd
import polars as pl

# Visualization
# import matplotlib.pyplot as plt

# NumPy settings
np.set_printoptions(precision=4)

# Pandas settings
pd.options.display.max_rows = 1_000
pd.options.display.max_columns = 1_000
pd.options.display.max_colwidth = 600

# Polars settings
pl.Config.set_fmt_str_lengths(1_000)
pl.Config.set_tbl_cols(n=1_000)
pl.Config.set_tbl_rows(n=200)

warnings.filterwarnings("ignore")

# Black code formatter (Optional)
%load_ext lab_black

# auto reload imports
%load_ext autoreload
%autoreload 2

In [2]:
from rich.console import Console
from rich.theme import Theme

custom_theme = Theme(
    {
        "white": "#FFFFFF",  # Bright white
        "info": "#00FF00",  # Bright green
        "warning": "#FFD700",  # Bright gold
        "error": "#FF1493",  # Deep pink
        "success": "#00FFFF",  # Cyan
        "highlight": "#FF4500",  # Orange-red
    }
)
console = Console(theme=custom_theme)


def create_path(path: str | Path) -> None:
    """
    Create parent directories for the given path if they don't exist.

    Parameters
    ----------
    path : str | Path
        The file path for which to create parent directories.
    """
    # Convert to Path object if it's a string
    path_obj: Path = Path(path) if isinstance(path, str) else path

    # Get the parent directory and create it if it doesn't exist
    path_obj.parent.mkdir(parents=True, exist_ok=True)


def go_up_from_current_directory(*, go_up: int = 1) -> None:
    """This is used to up a number of directories.

    Params:
    -------
    go_up: int, default=1
        This indicates the number of times to go back up from the current directory.

    Returns:
    --------
    None
    """
    import sys

    CONST: str = "../"
    NUM: str = CONST * go_up

    # Goto the previous directory
    prev_directory = os.path.join(os.path.dirname(__name__), NUM)
    # Get the 'absolute path' of the previous directory
    abs_path_prev_directory = os.path.abspath(prev_directory)

    # Add the path to the System paths
    sys.path.insert(0, abs_path_prev_directory)
    print(abs_path_prev_directory)

In [3]:
go_up_from_current_directory(go_up=2)

from settings import refresh_settings  # noqa: E402

settings = refresh_settings()

/Users/mac/Desktop/Projects/RAG-Tutorials


In [4]:
from langchain_openai import ChatOpenAI

model_str_remote: str = "google/gemini-2.0-flash-001"
model_str_local: str = "llama3.1:8b"  # llama3.1:8b, gemma3n:e4b, llama3.2:3b

# Deterministic responses
remote_llm = ChatOpenAI(
    api_key=settings.OPENROUTER_API_KEY.get_secret_value(),  # type: ignore
    base_url=settings.OPENROUTER_URL,
    temperature=0.0,
    model=model_str_remote,
)

local_llm = ChatOpenAI(
    api_key=settings.OLLAMA_API_KEY.get_secret_value(),
    base_url=settings.OLLAMA_URL,
    temperature=0.0,
    model=model_str_local,
)

<br>

## Vector Store And Embeddings

- TBC


### Create A QDRANT Client

In [5]:
from langchain_ollama import OllamaEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

client = QdrantClient(url=settings.QDRANT_URL)
embeddings = OllamaEmbeddings(
    model="mxbai-embed-large:latest",
)
emb = embeddings.embed_documents("Hello world")
emb_size: int = len(emb[0])

# Create a Qdrant collection (if it doesn't exist)
# client.create_collection(
#     collection_name="demo_collection",
#     vectors_config=VectorParams(size=emb_size, distance=Distance.COSINE),
# )

# Recreate the collection to ensure it's fresh
# client.recreate_collection(
#     collection_name="demo_collection",
#     vectors_config=VectorParams(size=emb_size, distance=Distance.COSINE),
# )

# # Create a vector store using Qdrant
# vector_store = QdrantVectorStore(
#     client=client,
#     collection_name="demo_collection",
#     embedding=embeddings,
# )

### Add Documents To The Vectorstore

In [6]:
# from uuid import uuid4

# from langchain_core.documents import Document

# docs: list[str] = [
#     "I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
#     "The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees Fahrenheit.",
#     "Building an exciting new project with LangChain - come check it out!",
#     "It is good to be good",
# ]
# all_metadata: list[dict[str, Any]] = [
#     {"source": "tweet"},
#     {"source": "news"},
#     {"source": "tweet"},
#     {"source": "tweet"},
# ]
# documents: list[Document] = [Document(page_content=doc, metadata=meta) for doc, meta in zip(docs, all_metadata)]
# uuids = [str(uuid4()) for _ in range(len(documents))]

# vector_store.add_documents(documents=documents, ids=uuids)

In [8]:
import bs4
from langchain import hub
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import OllamaEmbeddings
from langchain_openai import ChatOpenAI

embeddings = OllamaEmbeddings(
    model="mxbai-embed-large:latest",
)
#### INDEXING ####

# Load Documents
# loader = WebBaseLoader(
#     web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
#     bs_kwargs=dict(parse_only=bs4.SoupStrainer(class_=("post-content", "post-title", "post-header"))),
# )
loader = PyPDFLoader(file_path="../../data/chelsea_transfer_news.pdf")
docs = loader.load()

In [10]:
# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1_000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
len(splits)

25

In [11]:
console.print(docs[0])

Document(
    metadata={
        'producer': 'Skia/PDF m138',
        'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) 
Chrome/138.0.0.0 Safari/537.36',
        'creationdate': '2025-07-25T18:28:43+00:00',
        'title': 'Chelsea transfer news, rumours and gossip: Live updates and latest on deals, signings, loans and 
contracts | Football News | Sky Sports',
        'moddate': '2025-07-25T18:28:43+00:00',
        'source': '../../data/chelsea_transfer_news.pdf',
        'total_pages': 12,
        'page': 0,
        'page_label': '1'
    },
    page_content='Friday 25 July 2025 14:39, UK\nChelsea transfer news, rumours and\ngossip: Live updates and 
latest on\ndeals, signings, loans and contracts\nLatest Chelsea news\xa0\nSort by: Latest Oldest\nIn full: Chelsea 
2025/26 Premier League fixtures\xa0\nTransfer Centre LIVE! Deals, rumours, news on your phone\nDownload the Sky 
Sports app for Chelsea transfers, analysis and FREE highlights from\nEVERY Premier League game\xa0\xa0\xa0View 
post\n24 Jul\n16:22 Simons to Chelsea? The key questions answered...\nChelsea have held talks over signing RB 
Leipzig forward Xavi Simons - but how\nmuch will they have to pay and who else want him?\xa0\nSky Germany’s Leipzig
reporter Philipp Hinze answers the key questions\naround the deal.\xa0\nKeep scrolling!\xa0\nUPDATE\nF o o t b al l
\n News Watch Scores & FixturesTables Transfers More\n25/07/2025, 19:28 Chelsea transfer news, rumours and gossip: 
Live updates and latest on deals, signings, loans and contracts | Football News | Sky 
Sports\nhttps://www.skysports.com/football/live-blog/11668/13025497/chelsea-transfer-news-rumours-and-gossip-live-u
pdates-and-latest-on-deals-signings-loans-and-… 1/17'
)

In [16]:
from uuid import uuid4


collection_name: str = "demo_collection_2"

client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=emb_size, distance=Distance.COSINE),
)

# Create a vector store using Qdrant
vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name=collection_name,
    ids=[str(uuid4()) for _ in range(len(splits))],
)

In [13]:
prompt = hub.pull("rlm/rag-prompt")
console.print(prompt)

ChatPromptTemplate(
    input_variables=['context', 'question'],
    input_types={},
    partial_variables={},
    metadata={
        'lc_hub_owner': 'rlm',
        'lc_hub_repo': 'rag-prompt',
        'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'
    },
    messages=[
        HumanMessagePromptTemplate(
            prompt=PromptTemplate(
                input_variables=['context', 'question'],
                input_types={},
                partial_variables={},
                template="You are an assistant for question-answering tasks. Use the following pieces of retrieved 
context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences 
maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"
            ),
            additional_kwargs={}
        )
    ]
)

In [ ]:
llm = local_llm  # or remote_llm

retriever = vector_store.as_retriever()


def format_docs(docs: list[Any]) -> str:
    """Append the page content of each document into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | {"response": lambda x: x.content}
)

In [22]:
response = rag_chain.invoke("Who is Xavi?")
console.print(response)

{
    'response': 'Xavi is a football player who currently plays for RB Leipzig. He has been linked to transfers, 
with Chelsea reportedly leading the race and Bayern Munich also showing interest. Xavi sees himself as a leader at 
Leipzig and wants to take his development to the next level.'
}

In [23]:
response = rag_chain.invoke("Any news about Sterling?")
console.print(response)

{
    'response': "Fulham have expressed interest in signing Raheem Sterling from Chelsea this summer, as he is among
a 'bomb squad' of players surplus to requirements at Stamford Bridge. Sterling has two years remaining on his 
contract and was frozen out by Blues boss Enzo Maresca last season. He spent the previous season on loan at 
Arsenal."
}

In [24]:
response = rag_chain.invoke("What players are Chelsea FC interested in signing?")
console.print(response)

{
    'response': "Chelsea are interested in signing Xavi Simons from RB Leipzig and Jorrel Hato from Ajax, while 
also exploring a deal for Fulham's interest in Kiernan Dewsbury-Hall. Additionally, Chelsea have held talks to sign
Ajax's Hato. However, any further signings depend on exits from the club."
}

In [28]:
response = rag_chain.invoke("Any news about Bayern and Nkunku?")
console.print(response)

{
    'response': "Bayern Munich have been interested in signing Christopher Nkunku, but Inter Milan are also 
pursuing him. Bayern contacted Nkunku's side by phone to gather information, indicating they are still keeping a 
low profile in the background. Chelsea are currently leading the race for Nkunku."
}